<div style="padding: 20px; border-left: 5px solid #10b981; background-color: #f8fafc; border-radius: 8px;">

# ENA Dataset: YOLO Model
**Author:** OUFQUIR Khadija  
**Task:** Object Detection for Biodiversity Monitoring

---

### **Executive Summary**

This notebook documents the development of an object detection pipeline utilizing a **YOLO-based architecture** on the **ENA (European Nightjar Alliance)** dataset. The goal is to automate the identification of species in diverse environmental conditions.

The experimental workflow is divided into two primary phases:
- **Baseline Establishment:** Training a reference model with standard hyperparameters to evaluate initial performance benchmarks.
- **Extended Optimization:** Implementing continued training and hyperparameter fine-tuning, initiated from the baseline checkpoint, to explore potential gains in precision and model stability.

Key Findings:
Based on these trials, we observe that within current computational and data constraints, the model appears to have reached a performance plateau. This suggests that the results are primarily dictated by dataset characteristics—such as size and diversity—rather than a lack of optimization in the learning rate scheduling, augmentation, or regularization strategies.

### **Technical Stack**
**Framework:** `ultralytics` (YOLO)

**Dataset:** ENA Version 1.0

**Hardware:**   

- Baseline Training: NVIDIA A100 (Google Colab Pro)
- Hyperparameter Tuning: NVIDIA T4 (Google Colab Free Tier)
- Final Fine-Tuning: NVIDIA P100 / T4 (Kaggle Weekly GPU Quota)

</div>


---

# **I. Introduction & Project Scope**

In this notebook, we focus on the automated detection of avian species using the ENA dataset. Our primary objective is to leverage the YOLO (You Only Look Once) architecture to build a robust real-time detection system.

We have established a Baseline Model to serve as our performance benchmark. This initial phase helps us understand the data distribution, identify class imbalances, and set a "line in the sand" for future architectural optimizations and hyperparameter tuning.

# **II. Experimental Setup**

#### **1. Dataset Download Configuration**

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import drive

import json
import random
from collections import defaultdict
from pathlib import Path
import subprocess

import torch
import torch.utils.data
from torch.utils.data import Dataset, DataLoader, random_split

from PIL import Image


# ============================================
# 1) Mount Google Drive

drive.mount('/content/drive')


We define the source and destination paths for downloading the ENA24 dataset.

- **GCS_SOURCE**: This is the public Google Cloud Storage (GCS) path where the ENA24 images are hosted. It allows direct access to the dataset without manual download.

- **DRIVE_TARGET**: This is the local destination directory in Google Drive where the images will be stored. Saving the dataset in Google Drive ensures persistence across Colab sessions.

The target directory is created if it does not already exist, ensuring that the download process can proceed without errors.

In [ ]:

# Public GCP source
GCS_SOURCE = "gs://public-datasets-lila/ena24/images"

# Target folder in Drive
DRIVE_TARGET = "/content/drive/MyDrive/ena24/images"

os.makedirs(DRIVE_TARGET, exist_ok=True)
print("Dossier cible :", DRIVE_TARGET)

#### **2. Metadata Download**

This section downloads the metadata file associated with the ENA24 dataset.

- **META_SOURCE**: Specifies the public Google Cloud Storage (GCS) path where the metadata file is located. This file typically contains annotations and additional information about the dataset (e.g., labels, bounding boxes, categories).

- **META_TARGET**: Defines the destination directory in Google Drive where the metadata file will be stored. This ensures that the file is preserved across sessions.

In [ ]:
# download the metadata
# ============================================

META_TARGET = "/content/drive/MyDrive/ena24/metadata"
os.makedirs(META_TARGET, exist_ok=True)

META_SOURCE = "gs://public-datasets-lila/ena24/ena24.json"

cmd = [
    "gsutil",
    "cp",
    META_SOURCE,
    META_TARGET
]

result = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)
print("Code retour :", result.returncode)

#### **3. Full Dataset Download**

This code performs the download of the entire ENA24 image dataset (~3.6 GB) from the public Google Cloud Storage (GCS) bucket.

The download is performed using the `gsutil rsync` command with the following options:
- `-m`: Enables multi-threaded download for faster transfer.  
- `-r`: Copies directories recursively.

In [ ]:
# 3) Copy the entire dataset to Drive (3.6 GB)
# ============================================
import subprocess

cmd = [
    "gsutil",
    "-m",              # multi-thread
    "rsync",
    "-r",              # récursif
    GCS_SOURCE,
    DRIVE_TARGET
]

result = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

if result.returncode == 0:
    print("Téléchargement terminé avec succès.")
else:
    print("Erreur pendant le téléchargement. Code retour :", result.returncode)

#### **4. Copy Dataset to Local Colab Storage**

Working with a local copy can significantly improve read/write speed during preprocessing and training. The following script ensures that a consistent local copy of both the images and metadata is available for further processing.

- **DRIVE_IMAGES / DRIVE_JSON**: Paths in Google Drive where the original images and metadata are stored.  
- **LOCAL_IMAGES / LOCAL_JSON**: Paths in local Colab storage where the dataset will be copied.  

In [ ]:

DRIVE_ROOT = Path("/content/drive/MyDrive/ena24")
DRIVE_IMAGES = DRIVE_ROOT / "images"
DRIVE_JSON = DRIVE_ROOT / "metadata" / "ena24.json"

LOCAL_ROOT = Path("/content/ena24_local")
LOCAL_IMAGES = LOCAL_ROOT / "images"
LOCAL_JSON = LOCAL_ROOT / "ena24.json"

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

print("Source images :", DRIVE_IMAGES)
print("Source json   :", DRIVE_JSON)
print("Local root    :", LOCAL_ROOT)

# Copy local
if not LOCAL_IMAGES.exists():
    print("Copie des images vers le disque local Colab...")
    shutil.copytree(DRIVE_IMAGES, LOCAL_IMAGES)
    print("Images copiées.")
else:
    print("Images déjà présentes en local.")

if not LOCAL_JSON.exists():
    print("Copie du JSON vers le disque local Colab...")
    shutil.copy2(DRIVE_JSON, LOCAL_JSON)
    print("JSON copié.")
else:
    print("JSON déjà présent en local.")

In [ ]:
# Quick check
print("JSON exists:", LOCAL_JSON.exists())
print("Images exist:", LOCAL_IMAGES.exists())

n_imgs = sum(1 for p in LOCAL_IMAGES.rglob("*.jpg"))
print("Number of .jpg files found:", n_imgs)

#### **5. Dataset Preparation for YOLO**

Preparing our ENA24 dataset to be used with YOLO involves filtering categories, splitting into train/validation sets, and converting annotations into the YOLO format.

Configuration:
- **Filter categories:** We remove unwanted categories such as Humans and Vehicles. (They are not relevant for our wildlife detection task)
- **Validation ratio:** 20% of the dataset is used for validation.
- **Random seed:** Fixed for reproducibility.
- **Data directories:** We create folders for train/validation images and labels.

In [ ]:

# Config:
FILTER_CATEGORY_IDS = {8, 9}  # remove Human(8) and Vehicle(9). Set None to keep all
VAL_RATIO = 0.2
SEED = 42
NUM_WORKERS = 4

# Paths
yolo_root = LOCAL_ROOT / 'yolo_dataset'
img_train = yolo_root / 'images' / 'train'
img_val = yolo_root / 'images' / 'val'
lbl_train = yolo_root / 'labels' / 'train'
lbl_val = yolo_root / 'labels' / 'val'

for p in [img_train, img_val, lbl_train, lbl_val]:
    p.mkdir(parents=True, exist_ok=True)

The dataset comes with a COCO-format JSON file that contains image metadata and annotations. We create a mapping from image IDs to image info for easier access. Next, we filter the annotations to remove any objects that belong to categories we decided not to keep. Only annotations from relevant categories are retained. Images that have no valid annotations after filtering are discarded. This ensures that each image in the dataset has at least one object that the model can learn from, which is important to prevent errors during training.

YOLO requires numeric labels for each class, starting from 0. We assign a unique label to each category in the filtered dataset and also store the class names in order. This mapping is used both in the annotation files and in the configuration file, allowing the model to understand which class each bounding box belongs to.


In [ ]:
# Load COCO Metadata
with open(LOCAL_JSON, 'r') as f:
    coco = json.load(f)
# Build mappings
img_id_to_info = {img['id']: img for img in coco['images']}


# Filter Annotations & Categories
# annotations_by_image will store all annotations per image after filtering.
annotations_by_image = defaultdict(list)
for a in coco.get('annotations', []):
    if FILTER_CATEGORY_IDS is None or a['category_id'] not in FILTER_CATEGORY_IDS:
        annotations_by_image[a['image_id']].append(a)

cats = [
    c for c in coco.get('categories', [])
    if (FILTER_CATEGORY_IDS is None or c['id'] not in FILTER_CATEGORY_IDS)
]

# Maps COCO category IDs to YOLO numeric labels (0, 1, 2 …).
cat_ids = sorted({c['id'] for c in cats})
catid2label = {cid: idx for idx, cid in enumerate(cat_ids)}
label_names = [next(c['name'] for c in cats if c['id'] == cid) for cid in cat_ids]
print('Found classes:', catid2label)

# Keep only images that still have annotations
valid_images = [
    img for img in coco['images']
    if len(annotations_by_image.get(img['id'], [])) > 0
]
print('Images with annotations (after filter):', len(valid_images))


The valid images are shuffled and split into training and validation sets based on the defined validation ratio.

In [ ]:
# Train/val split
random.seed(SEED)
random.shuffle(valid_images)

n_total = len(valid_images)
n_val = int(n_total * VAL_RATIO)

train_images = valid_images[n_val:]
val_images = valid_images[:n_val]

- **COCO** annotations store bounding boxes as **[x, y, width, height]** in absolute pixel coordinates.
- **YOLO** requires **[class, x_center, y_center, width, height]** normalized between 0 and 1.

As consequence, to make the dataset compatible with YOLO training. for each image, we create a text file containing these normalized annotations. The images themselves are copied into the appropriate train or validation folders.

In [ ]:
# Convert to YOLO Format & Copy
def convert_and_copy(img_list, img_dest, lbl_dest):
    for img in img_list:
        src = LOCAL_IMAGES / img['file_name']
        if not src.exists():
            continue

        shutil.copy2(src, img_dest / img['file_name'])

        # get image size
        W, H = img.get('width'), img.get('height')
        if W is None or H is None:
            with Image.open(src) as im:
                W, H = im.size

        anns = annotations_by_image.get(img['id'], [])
        lines = []

        for a in anns:
            cid = a['category_id']
            if cid not in catid2label:
                continue

            cls = catid2label[cid]
            x, y, w, h = a['bbox']

            # convert to YOLO format
            x_c = (x + w / 2.0) / W
            y_c = (y + h / 2.0) / H
            w /= W
            h /= H

            lines.append(f'{cls} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}')

        # ✅ FIX 3: join with newline
        if lines:
            with open(lbl_dest / (Path(img['file_name']).stem + '.txt'), 'w') as f:
                f.write('\n'.join(lines))


# Run Conversion
convert_and_copy(train_images, img_train, lbl_train)
convert_and_copy(val_images, img_val, lbl_val)


Finally, we create a YAML configuration file for YOLO that contains the paths to the training and validation images, the number of classes, and the list of class names. This file is used directly during training and tells YOLO where to find the dataset and how to interpret the labels.

In [ ]:
# Write data.yaml for YOLO
data_yaml = yolo_root / 'data.yaml'

data = {
    'train': str(img_train.resolve()),
    'val': str(img_val.resolve()),
    'nc': len(catid2label),
    'names': label_names
}

with open(data_yaml, 'w') as f:
    import yaml
    yaml.dump(data, f)

print('Wrote', data_yaml)

In [ ]:
# Verification
from pathlib import Path

data_yaml = Path("/content/ena24_local/yolo_dataset/data.yaml")

print(data_yaml.read_text())

##**Environment & Baseline Configuration**

We use the Ultralytics package (YOLOv11). This cell installs the package and launches training. Training artifacts (checkpoints, plots) will be written under `runs/train` and we will copy the best checkpoint to Drive.

In [ ]:
# Install ultralytics (may take ~1-2 minutes)
!pip install -q "ultralytics>=8.3.0"

from ultralytics import YOLO
import glob, shutil, os

In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
from pathlib import Path
import torch

# Training hyperparameters (tune these)
BATCH_SIZE = 16
NUM_EPOCHS = 90
LR = 0.01
LRF = 0.01
WEIGHT_DECAY = 5e-4

# Paths
DATA_YAML = "/content/ena24_local/yolo_dataset/data.yaml"
PROJECT_DIR = "/content/drive/MyDrive/ena24_training"

# Créer dossier
Path(PROJECT_DIR).mkdir(parents=True, exist_ok=True)

# Vérifier si un checkpoint existe
last_ckpt = Path(PROJECT_DIR) / "train/weights/last.pt"

if last_ckpt.exists():
    print("🔁 Reprise de l'entraînement...")
    model = YOLO(str(last_ckpt))
    resume = True
else:
    print("🆕 Nouveau entraînement...")
    model = YOLO("yolo11s.pt")
    resume = False

# 🚀 Training
model.train(
    data=DATA_YAML,
    epochs=NUM_EPOCHS,
    batch=BATCH_SIZE,
    imgsz=640,
    workers=4,
    device=0 if torch.cuda.is_available() else "cpu",

    lr0=LR,
    lrf=LRF,     # LR == LRF == 0.01
    weight_decay=WEIGHT_DECAY,

    project=PROJECT_DIR,
    name="train",

    resume=resume,
    save_period=1
)

#**III. Results & Evaluation**
Plot training curves (loss, mAP) and display a few sample predictions using the trained model.

In [ ]:
# 1. Monter Google Drive
#from google.colab import drive
#drive.mount('/content/drive')

# 2. Imports
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from PIL import Image
from ultralytics import YOLO


# 3. Paths (IMPORTANT)
# =========================
BASE_DIR = Path("/content/drive/MyDrive/ena24_training/train")
WEIGHTS_DIR = BASE_DIR / "weights"

BEST_CKPT = WEIGHTS_DIR / "best.pt"
LAST_CKPT = WEIGHTS_DIR / "last.pt"
RESULTS_CSV = BASE_DIR / "results.csv"

print("BASE_DIR:", BASE_DIR)
print("BEST:", BEST_CKPT)
print("LAST:", LAST_CKPT)

## **1. mAP**



In [ ]:
if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)

    # Metrics curves (mAP)
    # ---------------------------
    plt.figure(figsize=(10, 5))

    if 'metrics/mAP50(B)' in df.columns:
        plt.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50', marker='o')
    if 'metrics/mAP50-95(B)' in df.columns:
        plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95', marker='x')

    plt.xlabel("Epoch")
    plt.ylabel("mAP")
    plt.title("YOLO Metrics")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("❌ results.csv not found")

### **Observation and Interpretation**

During the first 20 epochs, the model shows a steep increase in performance. This indicates that it quickly learns the most distinctive and easily recognizable features of the objects, such as characteristic shapes or patterns.

After approximately epoch 40, the performance curves become stable and relatively flat. This suggests that the model has reached a stable region of convergence and is mainly performing fine adjustments rather than learning new major patterns.

**Gap between mAP50 and mAP50-95:**  
- **mAP50 (~0.98):** This high value indicates that the model is very effective at detecting and localizing objects.  
- **mAP50-95 (~0.78):** This metric is more strict, as it evaluates performance across multiple IoU thresholds. The lower value compared to mAP50 reflects the increased difficulty of achieving precise bounding box alignment.  

Overall, this gap is expected, especially in wildlife datasets where object boundaries can be ambiguous or partially occluded. A score around 0.78 remains a strong result under these conditions.

  ## **2. Confusion matrix**

In [ ]:
from IPython.display import Image, display
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# path of normalized confusion matrix
conf_png = Path("/content/drive/MyDrive/ena24_training/train/confusion_matrix_normalized.png")

# display
display(Image(filename=str(conf_png)))

### **Observation and Interpretation**

A closer look at the confusion matrix shows that the model's main failure mode is **Type II Errors (False Negatives)** rather than **Type I Errors (False Positives)**. In simpler terms, the model is more likely to **miss an animal that is actually there** than to imagine an animal where there is none.  

- **Bottom Row (Background on Y-axis)**: These are the **False Positives**, where the **true label was Background** but the model predicted an animal. Thankfully, these values are very low ($0.01$ to $0.06$), meaning the model rarely “hallucinates” animals in empty frames.  
- **Far-Right Column (Background on X-axis)**: These are the **False Negatives**, where the **true label was an animal** (e.g., Eastern Chipmunk) but the model predicted Background. This indicates missed detections, particularly for smaller or camouflaged species.  

Overall, the normalized confusion matrix confirms that the model performs well on most classes, achieving near-perfect identification for larger mammals and clearly distinct species such as the Eastern Cottontail and Striped Skunk.  

However, a recurring pattern emerges with **Background False Negatives**. Smaller species or those that blend into the environment—like the Eastern Chipmunk (missed 17% of the time) and Wild Turkey (missed 12% of the time)—are sometimes classified as Background. This suggests that while the model rarely confuses one species for another, it struggles with **small-scale localization** or discards detections when the confidence score is low, possibly interpreting them as environmental noise.

The confusion matrix shown here comes from **best.pt**, the weights that achieved the highest mAP on the validation set. While this peak score looks impressive, it doesn’t automatically mean the model hasn’t overfitted. To really understand how well the model generalizes, we need to go beyond a single snapshot and look at the bigger picture—comparing the **Training vs. Validation Loss curves over time**.

  ## **3. Loss Curve Analysis**

  YOLO breaks down the loss into **3 main components**:

| Loss | Purpose | How to interpret it |
|------|---------|--------------------|
| **box_loss** | Error in bounding box localization | Lower = better box precision |
| **cls_loss** | Error in classification (labels) | Lower = better object class assignment |
| **dfl_loss** (Distribution Focal Loss) | Refines the exact bounding box size (bounding box regression) | Lower = more accurate object size and position |

- **train/xxx_loss** are computed on the training batch  
- **val/xxx_loss** are computed on the validation batch (or validation images)  


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    print("CSV preview:")
    display(df.head())

    # Loss curves
    # ---------------------------
    plt.figure(figsize=(10, 5))

    # Train Losses
    plt.plot(df['epoch'], df['train/box_loss'], label='train box_loss', linestyle='-')
    plt.plot(df['epoch'], df['train/cls_loss'], label='train cls_loss', linestyle='-')
    plt.plot(df['epoch'], df['train/dfl_loss'], label='train dfl_loss', linestyle='-')

    # Val Losses
    plt.plot(df['epoch'], df['val/box_loss'], label='val box_loss', linestyle='--')
    plt.plot(df['epoch'], df['val/cls_loss'], label='val cls_loss', linestyle='--')
    plt.plot(df['epoch'], df['val/dfl_loss'], label='val dfl_loss', linestyle='--')

    plt.xlabel("Epoch")
    plt.ylabel("Loss Value")
    plt.title("YOLO Training Losses")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("❌ results.csv not found")

### **Observation and Interpretation**

**1. Training and Validation Losses**

The evolution of loss functions provides a deep look into the model's learning behavior. A key observation is the synchronization between training and validation curves, which suggests a robust generalization to the ENA24 dataset.
- **Box Loss (Localization):** The continuous decrease in training loss indicates that the model is improving its ability to localize objects in the training set. However, the validation loss stabilizes after a certain number of epochs, which suggests that the model reaches a **saturation point in generalization performance.** The small gap between training and validation losses indicates no severe overfitting in localization, but rather a natural convergence behavior.
- **Class Loss (Classification):** We observe a sharp initial drop, indicating that the model quickly learned to distinguish between categories. By epoch 80, the loss reaches a low plateau, signifying high confidence in label assignment. The strong decrease in training loss shows that the model learns to classify training samples very effectively. However, the validation loss stabilizes earlier, indicating that classification performance on unseen data has reached its limit. This gap between training and validation losses suggests a **mild specialization to training data**, but it does not negatively impact final mAP performance.
- **DFL Loss (Distribution Focal Loss):** Training DFL loss decreases from approximately 1.15 to 0.9, while Validation DFL loss stabilizes around 1.0. This indicates that the model improves its fine localization ability during training, but validation performance becomes stable earlier. This behavior is expected when the model reaches the limit of spatial precision supported by the dataset.

**2. The "Closing Phase" Effect (Epoch 80-90)**

A distinct "step" or sudden drop is visible in the loss curves around epoch 80. This is a programmed behavior in the YOLO training pipeline:
- **Mosaic Augmentation Closure:** During the first 80 epochs, the model uses "Mosaic Augmentation" (combining 4 images into 1) to expose the model to complex spatial contexts.
- **Stabilization:** In the final 10 epochs, this augmentation is disabled. The model then trains on "natural" unmixed images. This usually results in a sudden drop in training loss.

A key observation is that after approximately epoch 60–80, training losses continue to decrease while validation losses remain stable.This pattern indicates that the model is still optimizing itself on the training set, but no longer improves generalization performance.

We also observe a decoupled convergence behavior between localization (Box/DFL) and classification losses. While the geometric losses (Box/DFL) reached a plateau on the validation set relatively early, the classification loss continued to improve until epoch 60. This indicates that the model required more iterations to refine its semantic understanding of the ENA24 classes than it did for spatial localization.

However, since **mAP continues to increase and remains stable**, this behavior corresponds more to **convergence and saturation rather than severe overfitting.** and continuing the training was justified.

In [ ]:
import pandas as pd
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the path to your training results
# Update this path to point to your specific 'train' folder
results_path = '/content/drive/MyDrive/ena24_training/train/results.csv'

if os.path.exists(results_path):
    # Load the results
    df = pd.read_csv(results_path)

    # Clean column names (YOLO often adds leading/trailing spaces)
    df.columns = [c.strip() for c in df.columns]

    # YOLO defines the "best" model based on 'fitness'
    # Fitness = 0.1 * mAP50 + 0.9 * mAP50-95
    if 'metrics/mAP50(B)' in df.columns and 'metrics/mAP50-95(B)' in df.columns:
        df['fitness'] = 0.1 * df['metrics/mAP50(B)'] + 0.9 * df['metrics/mAP50-95(B)']

        # Find the row with the maximum fitness
        best_row = df.loc[df['fitness'].idxmax()]

        print("--- Best Model Analysis ---")
        print(f"Best Epoch: {int(best_row['epoch'])}")
        print(f"mAP50 at this epoch: {best_row['metrics/mAP50(B)']:.4f}")
        print(f"mAP50-95 at this epoch: {best_row['metrics/mAP50-95(B)']:.4f}")
        print(f"Calculated Fitness: {best_row['fitness']:.4f}")
    else:
        print("Error: Required metric columns not found in results.csv")
else:
    print(f"Error: results.csv not found at {results_path}")

### **Showing model results on validation images**

In [ ]:
# Load model
# =========================
if BEST_CKPT.exists():
    print("✅ Loading BEST model")
    model = YOLO(str(BEST_CKPT))
elif LAST_CKPT.exists():
    print("⚠️ Loading LAST model")
    model = YOLO(str(LAST_CKPT))
else:
    raise FileNotFoundError("❌ No checkpoint found")


# Error analysis + grid display
# =========================
IMG_VAL_DIR = Path("/content/ena24_local/yolo_dataset/images/val")
LBL_VAL_DIR = Path("/content/ena24_local/yolo_dataset/labels/val")

if not IMG_VAL_DIR.exists():
    print("❌ Validation images folder not found:", IMG_VAL_DIR)

else:
    imgs = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        imgs.extend(IMG_VAL_DIR.glob(ext))

    if len(imgs) == 0:
        print("❌ No images found")

    else:
        print(f" {len(imgs)} images found")

        error_images = []

        for img_path in imgs:

            # Run prediction
            results = model.predict(
                source=str(img_path),
                imgsz=640,
                conf=0.25,
                verbose=False
            )
            r = results[0]

            # Number of predicted objects
            num_preds = len(r.boxes)

            # Load ground truth labels
            label_path = LBL_VAL_DIR / (img_path.stem + ".txt")
            num_gt = 0

            if label_path.exists():
                with open(label_path) as f:
                    num_gt = len(f.readlines())

            # Simple failure detection logic
            failure = False

            # Case 1: object exists but not detected (False Negative)
            if num_gt > 0 and num_preds == 0:
                failure = True

            # Case 2: detection exists but no ground truth (False Positive)
            elif num_gt == 0 and num_preds > 0:
                failure = True

            # Case 3: mismatch in number of objects
            elif abs(num_gt - num_preds) > 0:
                failure = True

            # Store failed cases
            if failure:
                error_images.append(r.plot())

            # Limit to 20 images
            if len(error_images) >= 9:
                break

        # =========================
        # DISPLAY AS GRID
        # =========================
        if len(error_images) == 0:
            print("✅ No errors detected in this sample")
        else:
            import matplotlib.pyplot as plt

            n = len(error_images)
            cols = 3
            rows = (n + cols - 1) // cols

            plt.figure(figsize=(15, 3 * rows))

            for i, img in enumerate(error_images):
                plt.subplot(rows, cols, i + 1)
                plt.imshow(img)
                plt.axis('off')

            plt.suptitle("Images where the model fails", fontsize=14)
            plt.tight_layout()
            plt.show()

In [ ]:
# 6. Error analysis + grid display
if not IMG_VAL_DIR.exists():
    print("❌ Validation images folder not found:", IMG_VAL_DIR)

else:
    imgs = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        imgs.extend(IMG_VAL_DIR.glob(ext))

    if len(imgs) == 0:
        print("❌ No images found")

    else:
        print(f"{len(imgs)} images found")

        # Shuffle to get different images each run
        import random
        random.shuffle(imgs)

        error_images = []

        for img_path in imgs:

            # Run prediction
            results = model.predict(
                source=str(img_path),
                imgsz=640,
                conf=0.25,
                verbose=False
            )
            r = results[0]

            # Number of predicted objects
            num_preds = len(r.boxes)

            # Load ground truth labels
            label_path = LBL_VAL_DIR / (img_path.stem + ".txt")
            num_gt = 0
            if label_path.exists():
                with open(label_path) as f:
                    num_gt = len(f.readlines())

            # Simple failure detection logic
            failure = False
            if num_gt > 0 and num_preds == 0:        # False negative
                failure = True
            elif num_gt == 0 and num_preds > 0:      # False positive
                failure = True
            elif abs(num_gt - num_preds) > 0:        # mismatch
                failure = True

            # Store failed cases
            if failure:
                error_images.append(r.plot())

            # Limit to 20 images
            if len(error_images) >= 9:
                break

        # Display as grid
        if len(error_images) == 0:
            print("✅ No errors detected in this sample")
        else:
            import matplotlib.pyplot as plt

            n = len(error_images)
            cols = 3
            rows = (n + cols - 1) // cols

            plt.figure(figsize=(15, 3 * rows))

            for i, img in enumerate(error_images):
                plt.subplot(rows, cols, i + 1)
                plt.imshow(img)
                plt.axis('off')

            plt.suptitle("Images where the model fails", fontsize=14)
            plt.tight_layout()
            plt.show()

### **Conclusion :**

The training process of the YOLO model demonstrates a stable convergence, characterized by a consistent decrease in loss functions across both training and validation phases. The close alignment between the training and validation curves indicates a high level of generalization and an absence of significant overfitting. However, a detailed examination reveals a performance plateau, particularly regarding the validation Box Loss and the Distribution Focal Loss (DFL), which suggests an underfitting of fine spatial details.

This stagnation indicates that while the model has successfully captured the primary features for classification, its ability to achieve high-precision localization is currently limited. The primary constraints appear to stem from the dataset's complexity and annotation quality rather than the training architecture itself.

To address these limitations, several areas for improvement are proposed:
- **Data quality:** Auditing and cleaning annotations (poorly fitted bounding boxes) is strongly recommended, as is increasing the dataset size by 20 to 50%, focusing on challenging cases.
- **Technical configuration:** Increasing the input resolution (from 640 to 768 or 1024) and using a more powerful model (such as YOLOv8m or YOLOv8l) could improve spatial accuracy and learning capacity.
- **Training optimization:** Adjusting hyperparameters, particularly through longer fine-tuning or using a cosine decay for the learning rate, is suggested to refine convergence.

Due to dataset size and ressources limitation, we will only focus on training optimization.

# **IV. Hyperparameter Tuning**

Hyperparameter tuning consists of testing several combinations of training parameters (learning rate, optimizer settings, data augmentation parameters etc.) in order to find the configuration that produces the best results. Instead of manually selecting these values, the tuning process automatically evaluates different configurations and compares their performance.

However, performing hyperparameter tuning on a large dataset can be very time-consuming, especially when using limited computational resources.To make the tuning process feasible, a **fast tuning strategy** was used.

### **Fast Tuning Strategy**

During the tuning phase, the goal is not to obtain the final model but to quickly identify promising hyperparameter values. For this reason, the following adjustments were applied:
- The number of epochs was reduced.
- The number of iterations (number of experiments) was limited.
- A smaller model (YOLO11n) was used to speed up training.

This approach significantly reduces the computational cost of tuning while still allowing the algorithm to explore different parameter combinations.

Once the best hyperparameters were identified, the **final training** can be performed using a **larger model and a higher number of epochs** to obtain the best possible performance.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

# 📂 Paths
DATA_YAML = "/content/ena24_local/yolo_dataset/data.yaml"
PROJECT_DIR = "/content/drive/MyDrive/ena24_training"

# 📁 dossier tuning
TUNE_DIR = Path(PROJECT_DIR) / "tuning"
TUNE_DIR.mkdir(parents=True, exist_ok=True)

# 🧠 modèle neuf (IMPORTANT)
model = YOLO("yolo11n.pt")

# 🚀 Hyperparameter tuning
model.tune(
    data=DATA_YAML,

    epochs=8,          # court (chaque test)
    iterations=8,      # nombre de tests (cad entraînement complet du modèle avec une combinaison d’hyperparamètres.)

    imgsz=512,         # réduire la taille des images car complexité ≈ taille_image²
    batch=16,

    device=0 if torch.cuda.is_available() else "cpu",

    project=str(TUNE_DIR),
    name="tune_fast_YOLOn",

    optimizer="SGD",     # optionnel
    plots=True           # 📊 génère des graphiques
)

### **Observation and Interpretation**

| Iteration | Fitness | mAP50 | mAP50-95 | Precision | Recall |
|----------|--------|-------|----------|----------|--------|
| 1        | 0.564  | 0.806 | 0.564    | 0.75     | 0.74   |
| 2        | 0.582  | 0.846 | 0.582    | 0.82     | 0.77   |
| 4        | 0.603  | 0.844 | 0.603    | 0.82     | 0.77   |
| 7 (best) | 0.606  | 0.853 | 0.606    | 0.81     | 0.78   |


The results show a consistent improvement in performance across iterations, with the fitness score increasing from 0.564 to 0.606. This confirms that the hyperparameter tuning process was effective in identifying better configurations. The best results were obtained at iteration 7, with improved mAP50 and mAP50-95 scores.

It is important to note that these results were obtained using a fast tuning strategy with reduced epochs and image size. Therefore, they do not represent the final performance of the model, but rather serve to guide the selection of optimal hyperparameters.

Based on these results, the optimized hyperparameters will be used for the final training phase with full training settings in order to achieve improved performance.

# **V. Final training with optimized hyperparameters**

In [ ]:
from ultralytics import YOLO

# Définissez le chemin vers votre fichier data.yaml ici
DATA_YAML = "/content/ena24_local/yolo_dataset/data.yaml"
PROJECT_DIR = "/content/drive/MyDrive/ena24_training"
model = YOLO("yolo11s.pt")

model.train(
    data=DATA_YAML,
    epochs=100,
    batch=16,
    imgsz=640,

    cfg="/content/drive/MyDrive/ena24_training/tuning/tune_fast_YOLOn/best_hyperparameters.yaml",
    plots=True
    project=PROJECT_DIR,
    name="final_model"
)

During the final stage of the project, we encountered hardware resource limitations in Google Colab, where the exhaustion of available Compute Units prevented the completion of the final training cycles under the intended high-performance conditions.

To keep the project on schedule and preserve the integrity of the deep learning models, we migrated the computational workload to Kaggle Kernels, which provided suitable hardware to complete training without compromising performance or timelines. Although the live execution logs are not included in this notebook, all model weights, performance metrics, and diagnostic plots were synchronized and stored in Google Drive, and the final analysis presented in this report is therefore based on these verified cloud-stored results.

### **Results** :

**1. mAP**

Initially, the `model.tune()` function was run for 8 iterations to identify the optimal hyperparameters. The resulting model (`best.pt`) then served as the starting point for the final 80-epoch training on Kaggle. This explains why the performance metrics (mAP) start at a high level (~0.98) from the very first epoch of the final run.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Nouveau chemin direct vers ton CSV
RESULTS_CSV = Path("/content/drive/MyDrive/ena24_training/final_model/final_model_80ep/results.csv")

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)

    print("CSV preview:")
    display(df.head())

    # =========================
    # 1. Metrics (mAP)
    # =========================
    plt.figure(figsize=(10, 5))

    if 'metrics/mAP50(B)' in df.columns:
        plt.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50', marker='o')
    if 'metrics/mAP50-95(B)' in df.columns:
        plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95', marker='x')

    plt.xlabel("Epoch")
    plt.ylabel("mAP")
    plt.title("YOLO Metrics")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("❌ fichier resultat.csv introuvable")

**2. Loss curves**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

RESULTS_CSV = Path("/content/drive/MyDrive/ena24_training/final_model/final_model_80ep/results.csv")

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    plt.figure(figsize=(10, 5))

    plt.plot(df['epoch'], df['train/box_loss'], label='train box_loss')
    plt.plot(df['epoch'], df['train/cls_loss'], label='train cls_loss')
    plt.plot(df['epoch'], df['train/dfl_loss'], label='train dfl_loss')

    plt.plot(df['epoch'], df['val/box_loss'], label='val box_loss', linestyle='--')
    plt.plot(df['epoch'], df['val/cls_loss'], label='val cls_loss', linestyle='--')
    plt.plot(df['epoch'], df['val/dfl_loss'], label='val dfl_loss', linestyle='--')

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("YOLO Training Losses")
    plt.legend()
    plt.grid(True)
    plt.show()

else:
    print("❌ fichier resultat.csv introuvable")

The training curves show a stable and consistent convergence behavior across all three main loss components:

- Box loss (localization error) steadily decreases over epochs, indicating improved bounding box regression and more accurate object localization.
- Classification loss shows a clear downward trend, especially after mid-training, suggesting that the model is progressively learning more discriminative class representations.
- DFL loss (Distribution Focal Loss) also decreases smoothly, confirming improved quality in bounding box distribution modeling.

On the validation side, all losses remain relatively stable with slight fluctuations, which suggests good generalization without strong overfitting. The gap between training and validation losses remains controlled, particularly for box and classification losses.

# **VI. Comparaison between baseline and tuned model**

From a comparative perspective, the hyperparameter configuration—including learning rate schedule, augmentation strategy, and regularization terms such as mosaic, mixup, and weight decay—does not produce a significant deviation from the baseline results. The differences observed are marginal and fall within a range that does not indicate a meaningful performance gain.

Furthermore, it is important to note that these results were obtained within the constraints of currently available computational resources. While more exhaustive optimization (such as a large-scale grid search or genetic evolution of hyperparameters) could theoretically yield further improvements, the lack of high-performance computing power necessitated a more focused experimental scope. Consequently, this configuration represents an optimized trial under existing hardware limitations rather than an absolute upper bound of the model's potential.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Path to your file on Drive
RESULTS_CSV = Path("/content/drive/MyDrive/ena24_training/final_model/baseline_vs_tuned_comparisons/classwise_comparison.csv")

if not RESULTS_CSV.exists():
    print(f"Erreur : Le fichier est introuvable à l'adresse {RESULTS_CSV}")
else:
    # 1. Load data
    df = pd.read_csv(RESULTS_CSV)

    # 2. Calculate the overall mean for each model
    metrics = ["precision", "recall", "mAP50", "mAP50-95"]

    baseline_means = [df[f"{m}_baseline"].mean() for m in metrics]
    finetuned_means = [df[f"{m}_finetuned"].mean() for m in metrics]

    #3. Creating the Bar Chart
    x = np.arange(len(metrics))
    width = 0.35

    plt.figure(figsize=(12, 7))
    plt.bar(x - width/2, baseline_means, width, label='Baseline', color='#3498db')
    plt.bar(x + width/2, finetuned_means, width, label='Fine model', color='#e67e22')

    # Formatting
    plt.ylabel('Score (0-1)')
    plt.title('Overall Comparison: Baseline vs Final model (Class Average)')
    plt.xticks(x, [m.upper() for m in metrics])
    plt.ylim(0, 1.1)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.6)

    # Add the values ​​above the bars
    for i, val in enumerate(baseline_means):
        plt.text(i - width/2, val + 0.02, f'{val:.3f}', ha='center', fontsize=9)
    for i, val in enumerate(finetuned_means):
        plt.text(i + width/2, val + 0.02, f'{val:.3f}', ha='center', fontsize=9)

    plt.tight_layout()

    plt.savefig(RESULTS_CSV.parent / "plots/global_metrics_comparison.png")


    plt.show()

#**VII. Final Conclusion**

Overall, these results suggest that both the baseline and the tuned configuration are functionally equivalent in terms of detection performance under current experimental conditions. This implies that the primary limiting factor is unlikely to be the optimization strategy alone, but rather the dataset itself—particularly its size (8k images), diversity, and potential annotation quality constraints.

In this context, the model appears to have reached a near-saturation point under the current data regime. Continued training or tuning under these specific resource and data constraints mainly stabilizes or slightly refines existing representations without yielding substantial gains in generalization. Therefore, further significant improvements would likely require dataset-level enhancements or access to more powerful computational clusters for more intensive hyperparameter exploration.
